# Week 41, Day 1: Dynamic arrays

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w041d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
capacity, size, copies = 1, 0, 0
grew_at = []
for item in range(20):
    if size == capacity:
        copies += size          # move every item to a block twice as big
        capacity *= 2
        grew_at.append(size)
    size += 1
print(capacity, copies, grew_at)

In [ ]:
import array
import sys

numbers = list(range(1000))
packed = array.array("i", range(1000))      # "i": each slot is a C int
print(packed[10], len(packed), packed.itemsize, "bytes per item")
print(sys.getsizeof(numbers) > sys.getsizeof(packed))
try:
    packed.append("eleven")
except TypeError as err:
    print("TypeError:", err)

In [ ]:
import sys

items = []
last = sys.getsizeof(items)
for i in range(64):
    items.append(i)
    size = sys.getsizeof(items)
    if size != last:
        print(f"after {len(items):>2} appends the block grew to {size} bytes")
        last = size

In [ ]:
class DynamicArray:
    def __init__(self):
        self._size = 0
        self._capacity = 1
        self._slots = [None] * self._capacity      # a fixed block

    def __len__(self):
        return self._size

    def append(self, item):
        if self._size == self._capacity:
            self._resize(2 * self._capacity)
        self._slots[self._size] = item
        self._size += 1

    def _resize(self, new_capacity):
        new = [None] * new_capacity
        for i in range(self._size):                # copy every item: O(n)
            new[i] = self._slots[i]
        self._slots, self._capacity = new, new_capacity

    def __getitem__(self, i):
        if i < 0:
            i += self._size
        if not 0 <= i < self._size:
            raise IndexError("index out of range")
        return self._slots[i]

a = DynamicArray()
for word in ["ant", "bee", "cat", "dog", "eel"]:
    a.append(word)
    print(len(a), a._capacity, a._slots)
print(a[0], a[-1])

In [ ]:
import time

def timed(work, n):
    items = list(range(n))
    start = time.perf_counter()
    work(items)
    return time.perf_counter() - start

def from_end(items):
    while items:
        items.pop()

def from_front(items):
    while items:
        items.pop(0)

for n in [10_000, 20_000, 40_000]:
    end, front = timed(from_end, n), timed(from_front, n)
    print(f"n={n:>6}: pop() {end * 1000:6.2f} ms   pop(0) {front * 1000:7.2f} ms")

> **Key idea.** An array is a block of equal slots, so item `i` is found by arithmetic in O(1). A dynamic array keeps a length and a capacity; when full, it copies everything into a block twice the size, so `append` is O(1) amortised. Inserting or deleting at `i` shifts every later item, so it's O(n − i): cheap at the end, O(n) at the front. Grow when full; shrink to half only when a quarter full, to avoid thrashing. A Python list is a dynamic array of references; `array.array` packs numbers of one type.

### Exercise: A dynamic array

Write `DynamicArray` with a starting capacity of 1, doubling when full. It needs `append(item)`, `__len__`, `__getitem__` (negative indexes count from the end; out of range raises `IndexError`), a read-only property `capacity`, and a counter `copies`: how many items have been copied during resizes so far. Use `[None] * n` for each block and never `append` to it.

In [ ]:
class DynamicArray:
    def __init__(self):
        self._size = 0
        self._capacity = 1
        self._slots = [None]
        self.copies = 0

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import ast\ndef test_append():\n    \"appending\"\n    a = DynamicArray()\n    caps = []\n    for i in range(9):\n        a.append(i * 10)\n        caps.append(a.capacity)\n    assert caps == [1, 2, 4, 4, 8, 8, 8, 8, 16] and len(a) == 9 and a.copies == 1 + 2 + 4 + 8\n    assert a[0] == 0 and a[8] == 80 and a[-1] == 80 and a[-9] == 0\n    for bad in [9, -10, 100]:\n        try:\n            a[bad]\n            assert False, f\"a[{bad}] should raise IndexError\"\n        except IndexError:\n            pass\n    try:\n        a.capacity = 99\n        assert False, \"capacity should be read-only\"\n    except AttributeError:\n        pass\ndef test_fixed_blocks():\n    \"blocks are fixed\"\n    calls = [n for n in ast.walk(ast.parse(source)) if isinstance(n, ast.Call) and isinstance(n.func, ast.Attribute) and n.func.attr in (\"append\", \"extend\", \"insert\")\n             and isinstance(n.func.value, ast.Attribute) and n.func.value.attr == \"_slots\"]\n    assert not calls, \"don't append to _slots: write into its slots by index\"")

### Exercise: Setting, looping and showing

Add to the array (the starter has the core): `__setitem__` (same index rules), `__iter__` (the items in order, not the empty slots), `__repr__` like `DynamicArray(['a', 'b'])`, and `__contains__`.

In [ ]:
class DynamicArray:
    def __init__(self):
        self._size, self._capacity, self._slots = 0, 1, [None]

    def __len__(self):
        return self._size

    def append(self, item):
        if self._size == self._capacity:
            new = [None] * (2 * self._capacity)
            for i in range(self._size):
                new[i] = self._slots[i]
            self._slots, self._capacity = new, 2 * self._capacity
        self._slots[self._size] = item
        self._size += 1

    def _index(self, i):
        if i < 0:
            i += self._size
        if not 0 <= i < self._size:
            raise IndexError("index out of range")
        return i

    def __getitem__(self, i):
        return self._slots[self._index(i)]

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_more():\n    \"set, iterate, show\"\n    a = DynamicArray()\n    for x in \"abc\":\n        a.append(x)\n    a[1] = \"B\"\n    a[-1] = \"C\"\n    assert list(a) == [\"a\", \"B\", \"C\"] and repr(a) == \"DynamicArray(['a', 'B', 'C'])\" and \"B\" in a and None not in a\n    try:\n        a[3] = \"d\"\n        assert False\n    except IndexError:\n        pass\n    assert repr(DynamicArray()) == \"DynamicArray([])\" and list(DynamicArray()) == []")

### Exercise: Inserting with shifts

Write `insert(slots, size, i, item)` for a fixed block `slots` holding `size` items (with at least one free slot after them): shift items `i` to `size - 1` one place right, **starting from the end**, put `item` at `i`, and return the number of items moved. An `i` beyond `size` means "at the end". Then write `remove_at(slots, size, i)`, which removes the item at `i` by shifting the later items left, sets the freed last slot to `None`, and returns the removed item. Neither may change the length of `slots`.

In [ ]:
def insert(slots, size, i, item):
    ___

def remove_at(slots, size, i):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_insert():\n    \"inserting\"\n    s = [\"a\", \"b\", \"c\", \"d\", None, None]\n    assert insert(s, 4, 1, \"X\") == 3 and s == [\"a\", \"X\", \"b\", \"c\", \"d\", None]\n    assert insert(s, 5, 9, \"Z\") == 0 and s == [\"a\", \"X\", \"b\", \"c\", \"d\", \"Z\"]\n    t = [\"a\", None]\n    assert insert(t, 1, 0, \"first\") == 1 and t == [\"first\", \"a\"]\ndef test_remove():\n    \"removing\"\n    s = [\"a\", \"b\", \"c\", \"d\", None]\n    assert remove_at(s, 4, 1) == \"b\" and s == [\"a\", \"c\", \"d\", None, None]\n    assert remove_at(s, 3, 2) == \"d\" and s == [\"a\", \"c\", None, None, None] and len(s) == 5")

### Exercise: Pop with shrinking

Add `pop()` to the array: it removes and returns the last item (an empty array raises `IndexError("pop from empty array")`), clears its slot to `None`, and then, if the array is now **a quarter full or less** and its capacity is more than 1, halves the capacity (copying into a new block). The starter has the rest.

In [ ]:
class DynamicArray:
    def __init__(self):
        self._size, self._capacity, self._slots = 0, 1, [None]

    @property
    def capacity(self):
        return self._capacity

    def __len__(self):
        return self._size

    def _resize(self, new_capacity):
        new = [None] * new_capacity
        for i in range(self._size):
            new[i] = self._slots[i]
        self._slots, self._capacity = new, new_capacity

    def append(self, item):
        if self._size == self._capacity:
            self._resize(2 * self._capacity)
        self._slots[self._size] = item
        self._size += 1

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pop():\n    \"popping and shrinking\"\n    a = DynamicArray()\n    for i in range(8):\n        a.append(i)\n    caps = []\n    got = []\n    for _ in range(8):\n        got.append(a.pop())\n        caps.append(a.capacity)\n    assert got == [7, 6, 5, 4, 3, 2, 1, 0] and caps == [8, 8, 8, 8, 8, 4, 2, 1], caps\n    try:\n        a.pop()\n        assert False\n    except IndexError as err:\n        assert str(err) == \"pop from empty array\"\ndef test_no_thrash():\n    \"no thrashing at the boundary\"\n    a = DynamicArray()\n    for i in range(8):\n        a.append(i)\n    caps = []\n    for _ in range(3):\n        a.append(\"x\")\n        caps.append(a.capacity)\n        a.pop()\n        caps.append(a.capacity)\n    assert caps == [16, 16, 16, 16, 16, 16], caps")

### Exercise: Counting copies

Write `copies_for(n, grow)`: start with capacity 1 and 0 items, append `n` items, and whenever the array is full, copy all its items to a new block whose capacity is `grow(old_capacity)`. Return the total number of items copied. Then set `double`, `plus_ten` and `one_and_half` to the copies for 1,000 appends with `grow` as doubling, adding 10, and multiplying by 1.5 (rounded **up** to a whole number, and always at least one more).

In [ ]:
import math

def copies_for(n, grow):
    ___

double = ___
plus_ten = ___
one_and_half = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_copies():\n    \"counting\"\n    assert copies_for(0, lambda c: 2 * c) == 0 and copies_for(1, lambda c: 2 * c) == 0 and copies_for(9, lambda c: 2 * c) == 15\n    assert double == 1023 and plus_ten == 49600 and one_and_half == copies_for(1000, lambda c: max(c + 1, math.ceil(c * 1.5)))\n    assert one_and_half < plus_ten // 10")

### Bug hunt: Moved, but not copied

When the array grows, `_resize` makes the new block but forgets to copy the old items into it, so earlier items are lost: this prints `[None, None, 'c']`. Fix it so it prints `['a', 'b', 'c']`.

In [ ]:
class DynamicArray:
    def __init__(self):
        self._size, self._capacity, self._slots = 0, 1, [None]

    def _resize(self, new_capacity):
        new = [None] * new_capacity
        self._slots, self._capacity = new, new_capacity

    def append(self, item):
        if self._size == self._capacity:
            self._resize(2 * self._capacity)
        self._slots[self._size] = item
        self._size += 1

    def items(self):
        return [self._slots[i] for i in range(self._size)]

a = DynamicArray()
for x in "abc":
    a.append(x)
print(a.items())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"copy on resize\"\n    expect_output(\"['a', 'b', 'c']\")")

### Bug hunt: One slot too late

`append` grows only when the size is **more** than the capacity, so it tries to write one past the end of a full block and crashes with `IndexError`. Fix the condition so this prints `5 8`.

In [ ]:
class DynamicArray:
    def __init__(self):
        self._size, self._capacity, self._slots = 0, 1, [None]

    def append(self, item):
        if self._size > self._capacity:
            new = [None] * (2 * self._capacity)
            for i in range(self._size):
                new[i] = self._slots[i]
            self._slots, self._capacity = new, 2 * self._capacity
        self._slots[self._size] = item
        self._size += 1

a = DynamicArray()
for x in range(5):
    a.append(x)
print(a._size, a._capacity)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"grow when full\"\n    expect_output(\"5 8\")")

### Bug hunt: Shifting the wrong way

`insert` shifts items right starting from the **front**, so each move overwrites the item that should move next, and this prints `['a', 'X', 'b', 'b', 'b']`. Fix the loop so it prints `['a', 'X', 'b', 'c', 'd']`.

In [ ]:
def insert(slots, size, i, item):
    for j in range(i + 1, size + 1):
        slots[j] = slots[j - 1]
    slots[i] = item

slots = ["a", "b", "c", "d", None]
insert(slots, 4, 1, "X")
print(slots)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"from the end\"\n    expect_output(\"['a', 'X', 'b', 'c', 'd']\")")

### Puzzle: Growing the block

Arrange the lines into the method that moves a dynamic array to a bigger block. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# For a dynamic array that starts at capacity 1 and doubles when full:

p = ___   # its capacity after 100 appends
q = ___   # how many times it has moved to a bigger block after 100 appends
r = ___   # how many items insert(0, x) shifts in an array of 50 items
s = ___   # how many items pop() shifts in an array of 50 items
t = ___   # the cost of items[999_999] in a list of a million: "O(1)", "O(log n)" or "O(n)"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 128 and q == 7 and r == 50, \"p, q and r\"\n    assert s == 0 and t == \"O(1)\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a fuller array

Write `Array`, a dynamic array (start at capacity 1, double when full, shrink to half at a quarter full, never below capacity 1) using a fixed block of `[None] * n` slots. It needs:

- `append`, `pop()` (the last item) and `pop(i)` (shifting the rest left), `insert(i, item)` (any `i` from `-len` to `len`, like a list; beyond that, clamp to the ends, as `list.insert` does);
- `extend(items)`, which resizes **at most once**: to the smallest power of two that fits everything, if the current block is too small;
- `__len__`, `__getitem__`, `__iter__`, `__eq__` (with other `Array`s and with lists, item by item), `__repr__` like `Array([1, 2])`;
- `capacity` (a read-only property) and `resizes`, a count of every time the block changed size.

In [ ]:
class Array:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def make(items):\n    a = Array()\n    for x in items:\n        a.append(x)\n    return a\ndef test_basic():\n    \"basics\"\n    a = make([1, 2, 3])\n    assert len(a) == 3 and a[0] == 1 and a[-1] == 3 and list(a) == [1, 2, 3] and repr(a) == \"Array([1, 2, 3])\" and a == [1, 2, 3] and a == make([1, 2, 3])\n    assert a.capacity == 4 and a.resizes == 2\ndef test_insert_pop():\n    \"insert and pop\"\n    a = make(\"abcd\")\n    a.insert(0, \"S\")\n    a.insert(-1, \"Y\")\n    a.insert(99, \"E\")\n    a.insert(-99, \"F\")\n    assert a == [\"F\", \"S\", \"a\", \"b\", \"c\", \"Y\", \"d\", \"E\"]\n    assert a.pop(1) == \"S\" and a.pop() == \"E\" and a == [\"F\", \"a\", \"b\", \"c\", \"Y\", \"d\"]\n    for _ in range(5):\n        a.pop(0)\n    assert a == [\"d\"] and a.capacity == 2\n    try:\n        make([]).pop()\n        assert False\n    except IndexError:\n        pass\ndef test_extend():\n    \"extend resizes once\"\n    a = make([1])\n    before = a.resizes\n    a.extend(range(2, 101))\n    assert a.resizes == before + 1 and a.capacity == 128 and list(a) == list(range(1, 101))\n    a.extend([])\n    assert a.resizes == before + 1")

## Warm-up

Twenty appends into a block that starts with one slot and doubles when full: the block moved five times, and 31 items were copied in all, fewer than two copies per append. That small number is the whole secret of why `append` is fast, and tomorrow's lesson proves it holds for any number of appends.

## What an array is

An **array** is a block of memory divided into equal-sized slots, one after another. Because every slot is the same size, the computer can find slot `i` with one calculation, whatever `i` is:

```text
address of slot i  =  address of slot 0  +  i × slot size

 slot:     0       1       2       3       4
        +-------+-------+-------+-------+-------+
        |  ...  |  ...  |  ...  |  ...  |  ...  |
        +-------+-------+-------+-------+-------+
address: 1000    1008    1016    1024    1032        (8-byte slots)
```

That's why `items[i]` is **O(1)** (Week 17): reading slot 5 or slot 5,000,000 is the same single sum and one memory read. No other structure in this phase can beat it.

A Python list is an array of **references**: each slot holds the address of an object, and the objects themselves live elsewhere. That's how one list can hold an int, a string and another list, since every reference is the same size, whatever it points to. When every item is a number of the same kind, the `array` module stores the numbers themselves, packed side by side, which uses far less memory:

(The list's size doesn't even count the 1,000 int objects it points to.) The price of the packed array is that it only holds one kind of value. Phase 4 builds its structures from Python lists used as plain fixed blocks, which is close enough to the real thing to show every cost.

## Length and capacity

An array's block has a fixed size. A **dynamic array** can grow anyway, by keeping two numbers:

- its **length** (or size): how many slots hold items;
- its **capacity**: how many slots the block has.

When `length < capacity`, `append` puts the item in the next free slot: O(1). When the block is full, it must:

1. ask for a **new, bigger block** (commonly twice the size);
2. **copy** every item across: O(n);
3. let go of the old block, then append as usual.

You can watch a real list do this. `sys.getsizeof` reports the bytes the list's own block uses, which jumps each time it moves (the exact numbers differ between computers, and CPython grows by about an eighth rather than doubling, plus a little):

## Building one

Here's a dynamic array of our own. A Python list made once with `[None] * capacity`, and never appended to, stands in for a fixed block: the class only ever reads and writes its slots by index.

Notice that `__getitem__` checks against `_size`, not `_capacity`: slots beyond the length exist but hold nothing the user put there. Forgetting that check is how real programs in languages without Python's safety read garbage.

## Inserting and deleting

Appending touches only the end. Inserting at position `i` must first **shift** every item from `i` onwards one slot to the right, starting from the end so nothing is overwritten; deleting shifts them back left:

```text
insert "X" at 1:   [a, b, c, d, _]  ->  [a, b, b, c, d]  ->  [a, X, b, c, d]
                         move d, then c, then b, right; then write X
```

So the costs of a list's operations are:

| Operation | Cost | Why |
|---|---|---|
| `items[i]`, `items[i] = x`, `len(items)` | O(1) | one calculation |
| `items.append(x)` | O(1) amortised | occasionally O(n) to grow; tomorrow shows it averages out |
| `items.pop()` | O(1) | nothing to shift |
| `items.insert(i, x)`, `items.pop(i)`, `del items[i]` | O(n − i) | shift everything after `i` |
| `items.insert(0, x)`, `items.pop(0)` | O(n) | shift everything |
| `x in items`, `items.index(x)`, `items.remove(x)` | O(n) | look at each item |

Emptying from the end grows in step with n; emptying from the front grows about four times when n doubles: O(n²) in all. When you need to remove from the front often, a list is the wrong structure: Week 43's queues fix it.

## Shrinking

If a dynamic array grows when full, should it shrink when half empty? It seems tidy, but it's a trap. Picture an array exactly full at capacity 8: an `append` grows it to 16 (copying 8); a `pop` leaves it half full and shrinks it to 8 (copying 8); the next `append` grows it again... Each of these alternating operations costs O(n). This is called **thrashing**.

The fix is to leave a gap between the two thresholds: grow when **full**, but shrink (to half) only when **a quarter** full. After either kind of resize, the array is half full, so it takes many cheap operations before the next resize. (This rule is a standard result; you'll find it in Cormen, Leiserson, Rivest and Stein's *Introduction to Algorithms*, in the chapter on amortised analysis.)

## Practice

## Quiz

## Daily challenge

## Before you go

Look for a `pop(0)` or `insert(0, ...)` inside a loop in a program of yours. How long would it take with a million items? Tomorrow: amortised analysis, the proof that doubling makes `append` O(1) on average, and why growing by a fixed amount doesn't.